# Real Image Editing: DDIM Inversion + Null-text Optimization
Fully automated — just **Runtime > Run All**.

In [ ]:
# 1. Setup
!git clone https://github.com/beratcelik1/real-image-editing.git 2>/dev/null; cd real-image-editing && git pull
%cd real-image-editing
!pip install -q torch diffusers transformers accelerate safetensors Pillow numpy torchvision scikit-image tqdm matplotlib

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 2. HuggingFace login (uses Colab Secrets — add HF_TOKEN in key icon, left sidebar)
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("Logged in via Colab secret")
except Exception:
    import os
    if os.environ.get("HF_TOKEN"):
        login(token=os.environ["HF_TOKEN"])
        print("Logged in via env var")
    else:
        print("WARNING: No HF token found. Add 'HF_TOKEN' in Colab secrets (key icon, left sidebar).")

In [ ]:
# 3. Download sample image
!wget -q -O data/sample.jpg "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=512"
IMAGE_PATH = "data/sample.jpg"
PROMPT = "a photo of a cat looking at the camera"

# Hyperparameters
NUM_STEPS = 50
CFG_SCALE = 7.5
OPT_STEPS = 10
LR = 1e-2

print(f"Image: {IMAGE_PATH}")
print(f"Prompt: {PROMPT}")

In [ ]:
# 4. Load model and image
from src.utils import (
    load_sd_components, load_image, encode_image,
    decode_latent, get_text_embeddings, get_uncond_embeddings, get_device
)
from src.inversion import ddim_inversion, null_text_optimization, reconstruct

device = get_device()
print(f"Device: {device}")

print("Loading Stable Diffusion 2.1...")
unet, vae, text_encoder, tokenizer, scheduler = load_sd_components(device)
print("Model loaded!")

image = load_image(IMAGE_PATH)
latent = encode_image(image, vae, device)
text_emb = get_text_embeddings(PROMPT, tokenizer, text_encoder, device)
uncond_emb = get_uncond_embeddings(tokenizer, text_encoder, device)
print(f"Image: {image.size}, Latent: {latent.shape}, Text emb: {text_emb.shape}")

from IPython.display import display
display(image)

In [ ]:
# 5. Stage 1: DDIM Inversion (~1 min on A100)
print("Running DDIM inversion...")
latent_T, latents_trajectory = ddim_inversion(
    latent, text_emb, uncond_emb, unet, scheduler,
    num_steps=NUM_STEPS, cfg_scale=CFG_SCALE,
)
print(f"Inverted latent: {latent_T.shape}, Trajectory: {len(latents_trajectory)} steps")

In [ ]:
# 6. Stage 2: Null-text Optimization (~5 min on A100)
print("Running null-text optimization...")
null_embeddings = null_text_optimization(
    latents_trajectory, text_emb, uncond_emb, unet, scheduler,
    num_steps=NUM_STEPS, cfg_scale=CFG_SCALE,
    opt_steps=OPT_STEPS, lr=LR,
)
print(f"Optimized {len(null_embeddings)} null embeddings")

In [ ]:
# 7. Stage 3: Reconstruction (~1 min on A100)
print("Reconstructing...")
recon_latent = reconstruct(
    latent_T, null_embeddings, text_emb, unet, scheduler,
    num_steps=NUM_STEPS, cfg_scale=CFG_SCALE,
)
recon_image = decode_latent(recon_latent, vae)
print("Done!")

In [ ]:
# 8. Results
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

orig_np = np.array(image).astype(np.float32)
recon_np = np.array(recon_image).astype(np.float32)
psnr = peak_signal_noise_ratio(orig_np, recon_np, data_range=255.0)
ssim = structural_similarity(orig_np, recon_np, channel_axis=2, data_range=255.0)

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
axes[0].imshow(image)
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(recon_image)
axes[1].set_title(f"Reconstruction (PSNR={psnr:.1f}dB, SSIM={ssim:.3f})")
axes[1].axis("off")
diff = np.abs(orig_np - recon_np).mean(axis=2)
im = axes[2].imshow(diff, cmap="hot")
axes[2].set_title("Pixel Difference")
axes[2].axis("off")
plt.colorbar(im, ax=axes[2], fraction=0.046)
plt.tight_layout()
plt.savefig("outputs/reconstruction_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nPSNR: {psnr:.2f} dB | SSIM: {ssim:.4f}")
print(f"Target: PSNR > 30 dB, SSIM > 0.9")

In [ ]:
# 9. Save outputs for teammates
from pathlib import Path
Path("outputs").mkdir(exist_ok=True)
image.save("outputs/original.png")
recon_image.save("outputs/reconstruction.png")

torch.save({
    "null_embeddings": null_embeddings,
    "latent_T": latent_T,
    "latents_trajectory": latents_trajectory,
    "text_embeddings": text_emb,
    "prompt": PROMPT,
    "num_steps": NUM_STEPS,
    "cfg_scale": CFG_SCALE,
}, "outputs/inversion_cache.pt")

print("Saved: outputs/original.png, outputs/reconstruction.png, outputs/inversion_cache.pt")
print("inversion_cache.pt contains everything needed for P2P/PnP editing.")